# Pendulum Filter and Smoother

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. Here the nonlinearities are functions in the model, deterministic nodes, and the fallback computes them as well: no approximation method is chosen per node.

The model is the noisy pendulum of Särkkä's *Bayesian Filtering and Smoothing*, a standard test of nonlinear filters. The state is the angle $\theta$ and the angular velocity $\omega$, discretised with step $\Delta t$, and a sensor measures the horizontal position of the bob, $\sin\theta$:

$$\begin{aligned}
x_t &= \begin{pmatrix} \theta_t \\ \omega_t \end{pmatrix} \sim \mathcal{N}\left(f(x_{t-1}), Q\right), \qquad f(x) = \begin{pmatrix} \theta + \omega\,\Delta t \\ \omega - g \sin(\theta)\,\Delta t \end{pmatrix},\\
y_t &\sim \mathcal{N}(\sin\theta_t, r).
\end{aligned}$$

We filter it online, one observation at a time, three ways: with `fallback = LocalLaplace()`, with `fallback = NaturalGradientPropagation()`, and with RxInfer's unscented transform chosen by hand for both functions, the usual way. A particle filter is the reference. Then we smooth it, the whole series at once, with `fallback = ExpectationPropagation()` and `fallback = LaplacePropagation()`, against a particle smoother.

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra, StableRNGs
using Plots, PrettyTables, Printf

## Data

In [ ]:
const Δt, g = 0.05, 9.81
transition(x) = [x[1] + x[2] * Δt, x[2] - g * sin(x[1]) * Δt]
sine(x) = sin(x[1])

qc, r, T = 0.5, 0.1, 100
Q = qc * [Δt^3/3 Δt^2/2; Δt^2/2 Δt]

rng = StableRNG(7)
x_true = Vector{Vector{Float64}}(undef, T)
x_true[1] = [1.5, 0.0]
for t in 2:T
    x_true[t] = rand(rng, MvNormal(transition(x_true[t - 1]), Q))
end
y = [rand(rng, Normal(sine(x), sqrt(r))) for x in x_true]
θ_true = first.(x_true)

plot(θ_true; color = :black, label = "angle θₜ", xlabel = "t", size = (800, 350))
scatter!(y; color = :gray, markersize = 2, label = "observations yₜ of sin θₜ")
hline!([-π / 2, π / 2]; color = :gray, linestyle = :dot, label = "±π/2")

The pendulum starts at 1.5 rad and the process noise pumps energy into it: it swings past $\pm\pi/2$, where $\sin\theta$ turns back. There, one observation fits two angles, $\theta$ and $\pm\pi - \theta$, and only the dynamics tell them apart.

## The filter

Each step is a small model: the previous state's belief as a prior, the transition, with its output, the predicted state, named `x_pred`, and the observation. Both functions enter the model as they are; without an approximation method, they go to the fallback. Its message towards a function's output is the moment-matched Gaussian of the inputs pushed through it, and its message back towards an input is the output's message pulled back through the function. The belief about $x_t$, the transition's Gaussian times the pulled-back observation, is then projected onto a Gaussian.

In [ ]:
@model function pendulum_step(y, m, V, Q, r)
    x_prev ~ MvNormalMeanCovariance(m, V)
    x_pred := transition(x_prev)
    x ~ MvNormalMeanCovariance(x_pred, Q)
    s := sine(x)
    y ~ NormalMeanVariance(s, r)
end

carry = @autoupdates begin
    m, V = mean_cov(q(x))
end

initial = @initialization begin
    q(x) = MvNormalMeanCovariance([1.5, 0.0], diagm([0.1, 0.1]))
    μ(x_pred) = MvNormalMeanCovariance([1.5, 0.0], diagm([0.1, 0.1]))
end

function run_filter(; kwargs...)
    observations = from(y) |> map(NamedTuple{(:y,), Tuple{Float64}}, (v) -> (y = v,))
    engine = infer(;
        model = pendulum_step(Q = Q, r = r), datastream = observations,
        autoupdates = carry, initialization = initial,
        returnvars = (:x,), historyvars = (x = KeepLast(),), keephistory = T, kwargs...,
    )
    return engine
end

The filter asks for the belief about $x_t$ only. The beliefs about the previous state and the transition's output would need messages backwards through the transition. Those are log-densities: the pulled-back observation alone says nothing about the velocity, and no proper Gaussian represents it. `LocalLaplace()` projects each belief as a product of such messages, so the filter does not ask for them; the smoother of the last section does.

`NaturalGradientPropagation()` keeps one site per edge of $x_t$ instead (Lukashchuk et al., *Information Geometry of Message Passing*). The transition's message, a Gaussian, is its own site. The observation pulled back through `sine`, a log-density $\ell$, gets a Gaussian site, which moves by a damped heavy-ball step in natural parameters, $v \leftarrow \beta v + \alpha(\eta^\star - \eta)$ and $\eta \leftarrow \eta + v$, with `alpha = 0.5` and `beta = 0.2` by default, towards its target $\eta^\star = G(q)^{-1}\,\mathrm{Cov}_q[T(x_t), \ell(x_t)]$: the natural gradient of $\mathbb{E}_q[\ell]$ at the belief $q$ about $x_t$, with $T$ and $G$ the Gaussian's sufficient statistics and Fisher information, by Gauss–Hermite points over $q$. At the fixed point the site is its target, and $q$ the Gaussian closest to the filtering belief in $\mathrm{KL}(q \,\|\, p)$, the one conjugate-computation variational inference finds. The predicted state `x_pred` keeps a site too, for the transition's pushforward, a log-density whose target is its moment-matched Gaussian. A site steps against its cavity, here the message back from `x`'s side, which a filter never asks for; the initial message `μ(x_pred)` stands in for it, and the site reaches the same Gaussian whatever that seed. The other two filters do not read it.

A site moves one step per iteration, and in a filter the iterations run within each observation's update: `iterations = 30` gives each observation thirty steps. Each update starts from the site and the velocity the last observation left, a warm start; thirty steps settle the site at its new target to the scheme's tolerance, `tol = 1e-8` relative, which the report checks. With a few steps only, the site would lag behind the observations.

In [ ]:
laplace = run_filter(fallback = LocalLaplace())
ngmp = run_filter(fallback = NaturalGradientPropagation(), iterations = 30, report = true)
unscented = run_filter(algorithm = @algorithm begin
    transition() -> Unscented()
    sine() -> Unscented()
end)

angles(engine) = (first.(mean.(engine.history[:x])), [sqrt(cov(q)[1, 1]) for q in engine.history[:x]])
m_laplace, s_laplace = angles(laplace)
m_ngmp, s_ngmp = angles(ngmp)
m_unscented, s_unscented = angles(unscented)

The report of the natural-gradient filter, over its hundred observations, shows the two functions computed by the fallback in the direction the filter needs, and the sites: the observation's site on $x_t$ with its steps, and the predicted state's. Every site has settled:

In [ ]:
report_ngmp = ngmp.model.metadata[:report]
@assert isempty(RxInfer.diagnostics(report_ngmp)) # hide
@assert report_ngmp.approximations.variables[(:x, nothing)].sites.outcome.method === :natural_gradient # hide
report_ngmp

## The particle filter

A bootstrap particle filter with 20 000 particles: each particle moves with the dynamics, is weighted by the observation, and the cloud is resampled. The average weight at each step estimates $p(y_t \mid y_{1:t-1})$, and their product the evidence $p(y)$, which the smoother's free energy is compared with.

In [ ]:
function particle_filter(y; n = 20_000, rng = StableRNG(1))
    particles = rand(rng, MvNormal([1.5, 0.0], diagm([0.1, 0.1])), n)
    L = cholesky(Q).L
    means, sds, logevidence = zeros(length(y)), zeros(length(y)), 0.0
    for t in eachindex(y)
        if t > 1
            for j in 1:n
                particles[:, j] = transition(particles[:, j]) + L * randn(rng, 2)
            end
        end
        logw = [logpdf(Normal(sin(particles[1, j]), sqrt(r)), y[t]) for j in 1:n]
        w = exp.(logw .- maximum(logw))
        logevidence += maximum(logw) + log(sum(w) / n)
        w ./= sum(w)
        means[t] = sum(w .* particles[1, :])
        sds[t] = sqrt(sum(w .* (particles[1, :] .- means[t]) .^ 2))
        particles = particles[:, rand(rng, Categorical(w), n)]
    end
    return means, sds, logevidence
end

m_pf, s_pf, logevidence_pf = particle_filter(y)

## Results

In [ ]:
plt = plot(m_pf; ribbon = 2s_pf, fillalpha = 0.2, color = :gray, label = "particle filter (±2 sd)", xlabel = "t", ylabel = "angle θₜ", size = (900, 500), legend = :outerbottom, legendcolumns = 3)
plot!(plt, θ_true; color = :black, linewidth = 1, label = "true θₜ")
plot!(plt, m_unscented; color = :green, label = "unscented, by hand")
plot!(plt, m_laplace; color = :blue, label = "LocalLaplace()")
plot!(plt, m_ngmp; color = :red, linestyle = :dash, label = "NaturalGradientPropagation()")
hline!(plt, [-π / 2, π / 2]; color = :gray, linestyle = :dot, label = "±π/2")

In [ ]:
rms(a) = sqrt(sum(abs2, a) / length(a))
coverage(m, s) = sum(abs.(m - θ_true) .<= 2s) / T
rows = [
    "particle filter" rms(m_pf - θ_true) 0.0 coverage(m_pf, s_pf);
    "unscented, by hand" rms(m_unscented - θ_true) rms(m_unscented - m_pf) coverage(m_unscented, s_unscented);
    "NaturalGradientPropagation()" rms(m_ngmp - θ_true) rms(m_ngmp - m_pf) coverage(m_ngmp, s_ngmp);
    "LocalLaplace()" rms(m_laplace - θ_true) rms(m_laplace - m_pf) coverage(m_laplace, s_laplace)
]
@assert rms(m_ngmp - m_pf) < 0.1 && rms(m_unscented - m_pf) < 0.1 # hide
@assert rms(m_laplace - m_pf) > 1.0 # hide
@assert abs(rms(m_ngmp - θ_true) - rms(m_pf - θ_true)) < 0.05 # hide
pretty_table(rows; column_labels = ["", "RMSE against the truth", "RMS difference from the particle filter", "θₜ within ±2 sd"], formatters = [fmt__printf("%.3f", 2:4)], fit_table_in_display_horizontally = false)

`NaturalGradientPropagation()` follows the particle filter closely, as the unscented filter does, with no method chosen for either function. `LocalLaplace()` follows it while the swing goes out past $-\pi/2$, and is lost for good as it comes back:

In [ ]:
out = findfirst(<(-π / 2), m_pf)
back = findnext(>(-π / 2), m_pf, out)
lost = findfirst(t -> abs(m_laplace[t] - m_pf[t]) > 0.5, 1:T)
@assert !isnothing(lost) && abs(lost - back) <= 2 # hide
@assert all(t -> abs(m_ngmp[t] - m_pf[t]) < 0.5, 1:T) # hide
@printf("the angle goes past -π/2 at t = %d and comes back at t = %d; Laplace leaves the particle filter at t = %d\n", out, back, lost)

Near $-\pi/2$, an observation of $\sin\theta$ fits the angle and its mirror image $-\pi - \theta$ about equally, and on the way back the two are close. The belief about $x_t$ can then be skewed towards the mirror image, or have a second mode there. The Laplace method takes the Gaussian at the mode it finds and its curvature, and once that mode is on the wrong side of the turn, the next step starts from it: the filter swings the wrong way and never returns. `NaturalGradientPropagation()` fits the Gaussian over the whole belief: its site's target is an expectation over the belief, weighing the region the dynamics predict, so a skewed belief moves its mean a little rather than flipping it.

## Smoothing

A smoother asks for the belief about every state given all the observations, and so for the messages backwards through both functions, the log-densities the filter did without. Cavity propagation gives them a Gaussian. Under `fallback = ExpectationPropagation()`, every random variable keeps one Gaussian site per edge, and its beliefs are products of sites: its marginal, and each message it sends, the product of the sites of the other edges, called the edge's cavity. A message that arrives as a log-density is projected together with its cavity, and the cavity is divided out again; what is left is the edge's site. Against its cavity, the observation pulled back through `sine` is a belief about the whole state, the velocity included. The sites change as their cavities do, and the iterations sweep them to a fixed point.

`ExpectationPropagation()` projects by the moments of each tilted belief, the message times its cavity; `LaplacePropagation()` by its mode and curvature. The model is the whole series:

In [ ]:
@model function pendulum(y, Q, r)
    x[1] ~ MvNormalMeanCovariance([1.5, 0.0], diagm([0.1, 0.1]))
    s[1] := sine(x[1])
    y[1] ~ NormalMeanVariance(s[1], r)
    for t in 2:length(y)
        x[t] ~ MvNormalMeanCovariance(transition(x[t - 1]), Q)
        s[t] := sine(x[t])
        y[t] ~ NormalMeanVariance(s[t], r)
    end
end

smooth(fallback) = infer(
    model = pendulum(Q = Q, r = r), data = (y = y,), iterations = 30,
    fallback = fallback, free_energy = true, returnvars = (x = KeepEach(),), report = true,
)

ep = smooth(ExpectationPropagation())
lp = smooth(LaplacePropagation())

For comparison, the unscented transform chosen by hand on the same model, which RxInfer iterates from a wide initial message, twenty times:

In [ ]:
unscented_smoother = infer(
    model = pendulum(Q = Q, r = r), data = (y = y,), iterations = 20,
    algorithm = @algorithm(begin
        transition() -> Unscented()
        sine() -> Unscented()
    end),
    initialization = @initialization(begin
        μ(x) = MvNormalMeanCovariance([0.0, 0.0], diagm([10.0, 10.0]))
    end),
    returnvars = (x = KeepEach(),),
)

The report of the EP run shows the sites: each state's posterior is formed by expectation propagation, and the two functions are computed by the fallback in both directions.

In [ ]:
@assert isempty(RxInfer.diagnostics(ep.report)) && isempty(RxInfer.diagnostics(lp.report)) # hide
@assert all(record -> record.sites.projections > 0, (ep.report.approximations.variables[(:x, t)] for t in 1:T)) # hide
ep.report.approximations

The reference is a particle smoother: the bootstrap filter's particles at every step, then whole trajectories drawn backwards through them, each step weighted by the dynamics that lead to the state already drawn (forward filtering, backward simulation).

In [ ]:
function particle_smoother(y; n = 5_000, m = 500, rng = StableRNG(1))
    prior = MvNormal([1.5, 0.0], diagm([0.1, 0.1]))
    L, W = cholesky(Q).L, inv(Q)
    particles, logweights = zeros(2, n, T), zeros(n, T)
    current = rand(rng, prior, n)
    for t in 1:T
        if t > 1
            ancestors = rand(rng, Categorical(exp.(logweights[:, t - 1])), n)
            for j in 1:n
                current[:, j] = transition(particles[:, ancestors[j], t - 1]) + L * randn(rng, 2)
            end
        end
        particles[:, :, t] = current
        logw = [logpdf(Normal(sin(current[1, j]), sqrt(r)), y[t]) for j in 1:n]
        logweights[:, t] = logw .- (maximum(logw) + log(sum(exp, logw .- maximum(logw))))
    end
    predicted = mapslices(transition, particles; dims = 1)
    s1, s2 = zeros(T), zeros(T)
    for _ in 1:m
        j = rand(rng, Categorical(exp.(logweights[:, T])))
        for t in T:-1:1
            if t < T
                d = particles[:, j, t + 1] .- predicted[:, :, t]
                logw = logweights[:, t] .- vec(sum(d .* (W * d); dims = 1)) ./ 2
                w = exp.(logw .- maximum(logw))
                j = rand(rng, Categorical(w ./ sum(w)))
            end
            s1[t] += particles[1, j, t]
            s2[t] += particles[1, j, t]^2
        end
    end
    means = s1 ./ m
    return means, sqrt.(max.(s2 ./ m .- means .^ 2, 0.0))
end

m_ps, s_ps = particle_smoother(y)

## Results of the smoothers

In [ ]:
# The posteriors after the last iteration
smoothed(result) = (first.(mean.(last(result.posteriors[:x]))), [sqrt(cov(q)[1, 1]) for q in last(result.posteriors[:x])])
m_ep, s_ep = smoothed(ep)
m_lp, s_lp = smoothed(lp)
m_us, s_us = smoothed(unscented_smoother)

plt = plot(m_ps; ribbon = 2s_ps, fillalpha = 0.2, color = :gray, label = "particle smoother (±2 sd)", xlabel = "t", ylabel = "angle θₜ", size = (900, 500), legend = :outerbottom, legendcolumns = 3)
plot!(plt, θ_true; color = :black, linewidth = 1, label = "true θₜ")
plot!(plt, m_us; color = :green, label = "unscented, by hand")
plot!(plt, m_lp; color = :blue, label = "LaplacePropagation()")
plot!(plt, m_ep; color = :red, linestyle = :dash, label = "ExpectationPropagation()")
hline!(plt, [-π / 2, π / 2]; color = :gray, linestyle = :dot, label = "±π/2")

In [ ]:
F_ep, F_lp = last(ep.free_energy), last(lp.free_energy)
rows = [
    "particle smoother" rms(m_ps - θ_true) 0.0 coverage(m_ps, s_ps) -logevidence_pf;
    "unscented, by hand" rms(m_us - θ_true) rms(m_us - m_ps) coverage(m_us, s_us) NaN;
    "LaplacePropagation()" rms(m_lp - θ_true) rms(m_lp - m_ps) coverage(m_lp, s_lp) F_lp;
    "ExpectationPropagation()" rms(m_ep - θ_true) rms(m_ep - m_ps) coverage(m_ep, s_ep) F_ep
]
@assert rms(m_ep - m_ps) < 0.025 && rms(m_lp - m_ps) < 0.025 # hide
@assert rms(m_ep - m_ps) < rms(m_us - m_ps) # hide
@assert max(rms(m_ep - θ_true), rms(m_lp - θ_true)) < 0.55 * min(rms(m_pf - θ_true), rms(m_ngmp - θ_true)) # hide
us_errors = [rms(first.(mean.(xs)) - θ_true) for xs in unscented_smoother.posteriors[:x]] # hide
@assert 0.15 < minimum(us_errors[2:end]) && maximum(us_errors) < 0.4 && maximum(us_errors) - minimum(us_errors[2:end]) > 0.05 # hide
settled(result) = maximum(abs, first.(mean.(last(result.posteriors[:x]))) - first.(mean.(result.posteriors[:x][20]))) # hide
@assert settled(ep) < 1e-7 && settled(lp) < 1e-7 # hide
@assert abs(F_ep + logevidence_pf) < 0.2 && abs(F_lp + logevidence_pf) < 0.2 # hide
pretty_table(rows; column_labels = ["", "RMSE against the truth", "RMS difference from the particle smoother", "θₜ within ±2 sd", "free energy"], formatters = [fmt__printf("%.3f", 2:5)], fit_table_in_display_horizontally = false)

Both cavity schemes follow the particle smoother to within two hundredths of a radian, through the turn back past $-\pi/2$ where the `LocalLaplace()` filter was lost, and their error against the truth is about half the filters': each angle is now informed by the observations after it as well. `LaplacePropagation()` is not `LocalLaplace()` run on the whole series: each of its sites is fitted against the rest of the belief, its cavity, so a message is never projected on its own. Given the whole series, the beliefs are close to Gaussian, and `LaplacePropagation()`, which places each site at the mode of its tilted belief, is as close as `ExpectationPropagation()`. The unscented transform chosen by hand, iterated on the whole model, does not settle: from one iteration to the next its error against the truth rises and falls, roughly between 0.2 and 0.3 over the twenty iterations, so where it stops depends on the number of iterations. The cavity schemes settle: their means after 20 and after 30 iterations agree to $10^{-7}$.

The free energy of both is within 0.2 of the particle filter's estimate of $-\log p(y)$. Here the free energy is expectation propagation's: each node without rules takes its term from its cavities, the other terms are Bethe's, and at a fixed point it approximates $-\log p(y)$. It is an estimate, not a bound, so a lower value is not a better fit. It is computed once per iteration, and it settles as the sites do:

In [ ]:
plot(ep.free_energy; marker = :circle, color = :red, label = "ExpectationPropagation()", xlabel = "iteration", ylabel = "free energy", size = (800, 350))
plot!(lp.free_energy; marker = :circle, color = :blue, label = "LaplacePropagation()")
hline!([-logevidence_pf]; color = :black, linestyle = :dot, label = "-log p(y), particle filter")

## References

- S. Särkkä (2013). *Bayesian Filtering and Smoothing*. Cambridge University Press.
- N. J. Gordon, D. J. Salmond and A. F. M. Smith (1993). Novel approach to nonlinear/non-Gaussian Bayesian state estimation. *IEE Proceedings F*, 140(2), 107–113.
- S. J. Godsill, A. Doucet and M. West (2004). Monte Carlo smoothing for nonlinear time series. *Journal of the American Statistical Association*, 99(465), 156–168.
- T. P. Minka (2001). Expectation propagation for approximate Bayesian inference. *Proceedings of UAI*, 362–369.
- E. Eskin, A. J. Smola and S. V. N. Vishwanathan (2003). Laplace propagation. *Advances in NIPS 16*.